In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

world = [
    ('Afghanistan', 'Asia', 652230, 25500100, 20343000000),
    ('Albania', 'Europe', 28748, 2831741, 12960000000),
    ('Algeria', 'Africa', 2381741, 37100000, 188681000000),
    ('Andorra', 'Europe', 468, 78115, 3712000000),
    ('Angola', 'Africa', 1246700, 20609294, 100990000000)
]

world_schema = ["name", "continent", "area", "population", "gdp"]

world_df = spark.createDataFrame(data=world, schema=world_schema)

# Write an SQL query to report the name, population, and area of the big countries.
# big --  area of at least three million (i.e., 3000000 km2), or it has a population of at least twenty-five million (i.e., 25000000).

big_country = world_df.where((col("population") >= 25000000) | (col('area') >= 3000000))
display(big_country)

name,continent,area,population,gdp
Afghanistan,Asia,652230,25500100,20343000000
Algeria,Africa,2381741,37100000,188681000000


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

customer_data = [
    (1, 'Will', None),
    (2, 'Jane', None),
    (3, 'Alex', 2),
    (4, 'Bill', None),
    (5, 'Zack', 1),
    (6, 'Mark', 2)
]

customer_schema = ['id', 'name', 'refree_id']

customer = spark.createDataFrame(customer_data, customer_schema)

result = customer.filter((col('refree_id') != 2) | col('refree_id').isNull())
display(result)

id,name,refree_id
1,Will,null
2,Jane,null
4,Bill,null
5,Zack,1


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

customer_data = [(1, 'Joe'), (2, 'Henry'), (3, 'Sam'), (4, 'Max')]

order_data = [(1,3),(2,1)]

customer_schema = ['id', 'name']

order_schema = ['id', 'customer_id']

customers = spark.createDataFrame(customer_data, customer_schema)

orders = spark.createDataFrame(order_data, order_schema)

result = customers.join(orders, customers['id']==orders['customer_id'], 'leftanti')
display(result)

id,name
2,Henry
4,Max


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

empz_data = [(1,8),(2,8),(3,8),(4,7),(5,9),(6,9)]
empz_schema = ['id', 'team_id']

empz = spark.createDataFrame(data=empz_data, schema=empz_schema)

WindowSpec = Window.partitionBy('team_id')

result = empz.withColumn('team_size', count('id').over(WindowSpec)).drop('team_id').orderBy(asc("id"))

display(result)

id,team_size
1,3
2,3
3,3
4,1
5,2
6,2


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

schema = [
    "user_id_sender",
    "user_id_receiver",
    "date",
    "action"
]

data = [
    ("ad4943sdz", "948ksx123d", "2020-01-04", "sent"),
    ("ad4943sdz", "948ksx123d", "2020-01-06", "accepted"),
    ("dfdfxf9483", "9djjjd9283", "2020-01-04", "sent"),
    ("dfdfxf9483", "9djjjd9283", "2020-01-15", "accepted"),
    ("ffdfff4234234", "lpjzjdi4949", "2020-01-06", "sent"),
    ("fffkfld9499", "993lsldidif", "2020-01-06", "sent"),
    ("fffkfld9499", "993lsldidif", "2020-01-10", "accepted"),
    ("fg503kdsdd", "ofp049dkd", "2020-01-04", "sent"),
    ("fg503kdsdd", "ofp049dkd", "2020-01-10", "accepted"),
    ("hh643dfert", "847jfkf203", "2020-01-04", "sent"),
    ("r4gfgf2344", "234ddr4545", "2020-01-06", "sent"),
    ("r4gfgf2344", "234ddr4545", "2020-01-11", "accepted"),
]

fb_friend_requests = spark.createDataFrame(data, schema)
# fb_friend_requests.show()

a = fb_friend_requests.filter(col('action')=="sent")
b = fb_friend_requests.filter(col('action')=="accepted")

result = a.join(b, (a['user_id_sender']==b['user_id_sender']) & (a['user_id_receiver']==b['user_id_receiver']), 'inner').groupBy(b.date).agg(count('*').alias('acceptance_count'))

display(result)

date,acceptance_count
2020-01-06,1
2020-01-15,1
2020-01-10,2
2020-01-11,1


In [0]:
schema = [
    "customer_id",
    "city",
    "account_balance",
    "account_status"
]

data = [
    (101, "Mumbai", 12500.50, "Active"),
    (102, "Pune", 0.00, "Inactive"),
    (103, "Delhi", 45890.75, "Active"),
    (104, "Bengaluru", -250.00, "Overdrawn"),
    (105, "Hyderabad", 8900.00, "Active"),
    (106, "Chennai", 120000.25, "Premium"),
    (107, "Kolkata", 3400.80, "Active"),
    (108, "Ahmedabad", 0.00, "Dormant"),
    (109, "Jaipur", 780.50, "Active"),
    (110, "Lucknow", -1250.75, "Overdrawn"),
    (111, "Nagpur", 24500.00, "Premium"),
    (112, "Indore", 5600.40, "Active"),
    (113, "Surat", 150.00, "Inactive"),
    (114, "Bhopal", 99875.30, "Premium"),
    (115, "Visakhapatnam", 4500.00, "Active"),
    (116, "Nashik", 0.00, "Dormant"),
    (117, "Patna", 32500.90, "Active"),
    (118, "Chandigarh", -500.00, "Overdrawn"),
    (119, "Coimbatore", 17250.60, "Active"),
    (120, "Goa", 68500.00, "Premium")
]

customers = spark.createDataFrame(data, schema)
active_customers = customers.where(col('account_status')!='Inactive')

city_wise = active_customers.groupBy('city').agg(avg('account_balance').alias('avg_balance'), count('customer_id').alias('total_customers'))

result = city_wise.filter((col('avg_balance') > 50000) & (col('total_customers') > 0)).select('city', 'avg_balance', 'total_customers')
display(result)

city,avg_balance,total_customers
Chennai,120000.25,1
Bhopal,99875.3,1
Goa,68500.0,1
